#### Publish: send gold results to GitHub
Exports the current gold data into one JSON file and commits it to the project repo at
`docs/data/parkway_status.json`. GitHub Pages serves that file to the public map.

**Contents of the file**
- `summary`: current miles and percentages for ALL, VA, NC (KPI cards)
- `sections`: every section as a GeoJSON line with its status (map)

**Security:** the GitHub token is read from the Databricks secret `brp/github_token`; it never appears in code.

**Steps**
1. **Settings:** repo, file path, tables, secret names
2. **Load:** current gold sections and summary
3. **Build:** assemble the JSON payload
4. **Checks:** sections, scopes, and file size look right
5. **Publish:** commit the file to GitHub (skipped if nothing changed)

In [0]:
# Standard library
import json
import base64

# Third-party
import requests

GITHUB_OWNER  = "marcusdenice"             # <- change this
GITHUB_REPO   = "blue-ridge-parkway-closures"
GITHUB_BRANCH = "main"
FILE_PATH     = "docs/data/parkway_status.json"    # where the website reads it

SECRET_SCOPE = "brp"
SECRET_KEY   = "github_token"

GOLD_SECTIONS = "brp.gold.section_status"
GOLD_SUMMARY  = "brp.gold.daily_summary"           # or current_summary, if you renamed it

In [0]:
# ---------------------------------------------------------------------------
# LOAD THE CURRENT GOLD DATA
# ---------------------------------------------------------------------------

sections = spark.sql(f"""
    SELECT run_date, fetched_at, nps_as_of,
           state, section_order, mp_start, mp_end, length_mi, crossroads,
           status, status_raw, notes, gated, geometry_geojson
    FROM {GOLD_SECTIONS}
    ORDER BY section_order
""").collect()

summary_rows = spark.sql(f"SELECT * FROM {GOLD_SUMMARY}").collect()

assert sections, f"{GOLD_SECTIONS} is empty: run the gold notebook first"
print(f"Sections: {len(sections)} | Summary rows: {len(summary_rows)} | Run date: {sections[0]['run_date']}")

In [0]:
# ---------------------------------------------------------------------------
# BUILD THE JSON PAYLOAD
# Everything the website needs, in one file. Each section becomes a GeoJSON
# "Feature": its line (geometry) plus its details (properties).
# ---------------------------------------------------------------------------

features = []
for s in sections:
    features.append({
        "type": "Feature",
        "geometry": json.loads(s["geometry_geojson"]),   # stored as text in gold; turn back into JSON
        "properties": {
            "state":      s["state"],
            "order":      s["section_order"],
            "mp_start":   s["mp_start"],
            "mp_end":     s["mp_end"],
            "length_mi":  s["length_mi"],
            "crossroads": s["crossroads"],
            "status":     s["status"],
            "status_raw": s["status_raw"],
            "notes":      s["notes"],
            "gated":      s["gated"],
        },
    })

# asDict() turns a Spark Row into a plain dictionary; key the summary by scope (ALL / VA / NC)
summary = {r["scope"]: r.asDict() for r in summary_rows}

first = sections[0]   # run details are the same on every row (one run)
payload = {
    "run_date":   first["run_date"],
    "fetched_at": first["fetched_at"],
    "nps_as_of":  first["nps_as_of"],
    "summary":    summary,
    "sections":   {"type": "FeatureCollection", "features": features},
}

# default=str converts dates/timestamps (which JSON can't store directly) into text
content = json.dumps(payload, default=str, indent=1)
print(f"Payload size: {len(content) / 1024:.1f} KB")

In [0]:
# ---------------------------------------------------------------------------
# CHECKS BEFORE PUBLISHING
# ---------------------------------------------------------------------------

# 1. Every section made it into the file
assert len(features) == len(sections) > 0, "Check 1: no sections to publish"

# 2. Summary has the whole parkway and both states
assert set(summary) == {"ALL", "VA", "NC"}, f"Check 2: unexpected summary scopes {set(summary)}"

# 3. Summary and sections come from the same run
assert all(r["run_date"] == first["run_date"] for r in summary_rows), \
    "Check 3: summary and sections are from different runs; run the gold notebook"

# 4. File is small enough for GitHub's simple file API (1 MB limit)
assert len(content) < 900_000, f"Check 4: payload too large ({len(content)} bytes)"

print("Publish checks passed")

In [0]:
# ---------------------------------------------------------------------------
# COMMIT THE FILE TO GITHUB
# Uses GitHub's "contents" API: GET the current file (to learn its version
# id, "sha"), then PUT the new content. Skips the commit if nothing changed.
# ---------------------------------------------------------------------------

token = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)   # never printed

api_url = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{FILE_PATH}"
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

# 1. Is the file already there? If so, GitHub needs its sha to replace it
current = requests.get(api_url, headers=headers, params={"ref": GITHUB_BRANCH}, timeout=30)
if current.status_code == 200:
    existing = current.json()
    sha = existing["sha"]
    old_content = base64.b64decode(existing["content"]).decode("utf-8")
elif current.status_code == 404:
    sha, old_content = None, None          # first publish: the file doesn't exist yet
else:
    current.raise_for_status()

# 2. Nothing changed (e.g. re-running without a new download): skip, to avoid an empty commit
if old_content == content:
    print("No changes since the last publish; nothing committed")
else:
    body = {
        "message": f"data: publish parkway status for {first['run_date']}",
        "content": base64.b64encode(content.encode("utf-8")).decode("ascii"),
        "branch":  GITHUB_BRANCH,
    }
    if sha:
        body["sha"] = sha                  # required when updating an existing file

    response = requests.put(api_url, headers=headers, json=body, timeout=30)
    response.raise_for_status()
    print("Published:", response.json()["commit"]["html_url"])